# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrhman-Moubarak/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup

In [20]:
import duckdb
import pandas as pd
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

FACT_MARCH = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
DIM_CONTENT = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'

## 1. My rule and its reason codes

Started with content_age_days and average_search_position, the two flag-linked signals
named in the assignment. average_search_position was set aside first: 52.5% of rows had
no value, confirmed to match gsc_data_available = False exactly, a real tracking gap,
not corrupted data. Needed a second signal, so I tested total_clicks_h1, search_volume,
and content_type.

total_clicks_h1 (volume / quick-win): 83.8% of rows have zero clicks. Since is_declining
is defined as H2 clicks < H1 clicks, a zero-click row can never be labeled declining, so
its 0.0% result is a label artifact. The real comparison, low vs high clicks, showed
almost no gap (56.3% vs 55.2%, n=30,328 and n=21,578). Verdict: FALSE.

search_volume (volume / quick-win, alternative): 18.2% NULL plus 37.8% zero, and the
non-null values are extremely skewed (median 10, max 368,000). Bucket results were weak
and inconsistent (high 11.5%, low 12.3%, zero 9.2%). Verdict: MIXED.

content_age_days (staleness / refresh): 5.3% of rows had negative age, content created
inside or after the H1 window, excluded as undefined. On the remaining 94.7%, decline
rate fell as age rose: young 14.2% (n=102,493), mid 7.1% (n=100,458), old 6.2%
(n=99,741). Opposite of the assumption that old content decays. Verdict: OPPOSITE.

content_type: fully populated, healthy category sizes. Decline rate varied widely:
keyword article 10.5% (n=266,261), comparison article 5.0% (n=3,392), feedly article
1.5% (n=50,105). A real, sizeable spread, kept as the second signal after both
volume candidates came back weak.

**Kept: content_age_days and content_type.**

Rule, flag content as at-risk when it is young (bottom third of age)
and is a keyword article, the two conditions with the strongest observed decline rates.

Reason codes:
- YOUNG_KEYWORD_ARTICLE: young age bucket and keyword article type
- YOUNG_OTHER_TYPE: young age bucket, non-keyword-article type
- AGED_KEYWORD_ARTICLE: mid/old age bucket, still a keyword article
- LOW_RISK: mid/old age, non-keyword-article type

In [21]:
# H1 features: average position, content age, content type, total clicks, search volume
features_h1 = con.sql(f"""
    WITH h1 AS (
        SELECT
            content_hash_id,
            client_hash_id,
            AVG(gsc_avg_position) AS average_search_position,
            SUM(gsc_clicks)       AS total_clicks_h1
        FROM read_parquet('{FACT_MARCH}')
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        h1.*,
        DATE_DIFF('day', dc.content_created_date, DATE '2026-03-01') AS content_age_days,
        dc.content_type,
        dc.search_volume
    FROM h1
    JOIN read_parquet('{DIM_CONTENT}') dc USING (content_hash_id)
""").df()

label_h2 = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(gsc_clicks) AS total_clicks_h2
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY content_hash_id, client_hash_id
""").df()

frame = features_h1.merge(label_h2, on=["content_hash_id", "client_hash_id"], how="inner")
frame["is_declining"] = (frame["total_clicks_h2"] < frame["total_clicks_h1"]).astype(int)

print("=== Frame ===")
print("shape:", frame.shape)
print(frame["is_declining"].value_counts())

print("\n=== Engagement Rate (GA4 availability) ===")
ga4_check = con.sql(f"""
    SELECT ga4_data_available, COUNT(*) AS n_rows
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY ga4_data_available
""").df()
print(ga4_check)

print("\n=== Average Search Position ===")
print(frame["average_search_position"].describe())
n_missing_position = frame["average_search_position"].isna().sum()
print("missing position rows:", n_missing_position)
print("missing position pct:", n_missing_position / len(frame) * 100)

gsc_check = con.sql(f"""
    SELECT gsc_data_available, COUNT(*) AS n_rows
    FROM read_parquet('{FACT_MARCH}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY gsc_data_available
""").df()
print(gsc_check)

print("\n=== Content Age ===")
print(frame["content_age_days"].describe())
n_before = len(frame)
n_negative_age = (frame["content_age_days"] < 0).sum()
print("negative age rows:", n_negative_age)
print("negative age pct:", n_negative_age / n_before * 100)

print("\n=== Total Clicks H1 ===")
print(frame["total_clicks_h1"].describe())
n_zero_clicks = (frame["total_clicks_h1"] == 0).sum()
print("zero click rows:", n_zero_clicks)
print("zero click pct:", n_zero_clicks / len(frame) * 100)

print("\n=== Content Type ===")
print(frame["content_type"].value_counts(dropna=False))

print("\n=== Search Volume ===")
n_null_sv = frame["search_volume"].isna().sum()
n_zero_sv = (frame["search_volume"] == 0).sum()
print("null search_volume rows:", n_null_sv)
print("zero search_volume rows:", n_zero_sv)
print("total rows:", len(frame))
print(frame["search_volume"].describe())

print("\n=== Age Bucket Table ===")
frame_age_valid = frame[frame["content_age_days"] >= 0].copy()
print("rows used for age test:", len(frame_age_valid))
frame_age_valid["age_bucket"] = pd.qcut(frame_age_valid["content_age_days"], q=3,
                                         labels=["young", "mid", "old"])
age_table = frame_age_valid.groupby("age_bucket", observed=True)["is_declining"].agg(["mean", "count"])
age_table.columns = ["pct_declining", "n"]
print(age_table)

print("\n=== Click Bucket Table ===")
median_nonzero = frame.loc[frame["total_clicks_h1"] > 0, "total_clicks_h1"].median()
print("median of nonzero clicks:", median_nonzero)
frame["click_bucket"] = "no_clicks"
frame.loc[(frame["total_clicks_h1"] > 0) & (frame["total_clicks_h1"] <= median_nonzero), "click_bucket"] = "low_clicks"
frame.loc[frame["total_clicks_h1"] > median_nonzero, "click_bucket"] = "high_clicks"
click_table = frame.groupby("click_bucket")["is_declining"].agg(["mean", "count"])
click_table.columns = ["pct_declining", "n"]
print(click_table)

print("\n=== Search Volume Bucket Table ===")
frame_sv_valid = frame[frame["search_volume"].notna()].copy()
median_nonzero_sv = frame_sv_valid.loc[frame_sv_valid["search_volume"] > 0, "search_volume"].median()
print("median of nonzero search_volume:", median_nonzero_sv)
frame_sv_valid["sv_bucket"] = "zero_volume"
frame_sv_valid.loc[
    (frame_sv_valid["search_volume"] > 0) & (frame_sv_valid["search_volume"] <= median_nonzero_sv),
    "sv_bucket"
] = "low_volume"
frame_sv_valid.loc[frame_sv_valid["search_volume"] > median_nonzero_sv, "sv_bucket"] = "high_volume"
sv_table = frame_sv_valid.groupby("sv_bucket")["is_declining"].agg(["mean", "count"])
sv_table.columns = ["pct_declining", "n"]
print(sv_table)

print("\n=== Content Type Bucket Table ===")
type_table = frame.groupby("content_type")["is_declining"].agg(["mean", "count"])
type_table.columns = ["pct_declining", "n"]
print(type_table.sort_values("pct_declining", ascending=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Frame ===
shape: (319758, 9)
is_declining
0    290769
1     28989
Name: count, dtype: int64

=== Engagement Rate (GA4 availability) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   ga4_data_available   n_rows
0                <NA>  1761384
1               False  2721811
2                True   159060

=== Average Search Position ===
count    151980.000000
mean         15.653035
std          17.658603
min           0.000000
25%           4.817037
50%           8.285743
75%          19.753293
max         310.000000
Name: average_search_position, dtype: float64
missing position rows: 167778
missing position pct: 52.47030566866192


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_data_available   n_rows
0               False  3002018
1                True  1640237

=== Content Age ===
count    319758.000000
mean        185.124141
std         116.339984
min         -19.000000
25%          80.000000
50%         195.000000
75%         251.000000
max         464.000000
Name: content_age_days, dtype: float64
negative age rows: 17066
negative age pct: 5.337161228178811

=== Total Clicks H1 ===
count    319758.000000
mean          1.203451
std           9.215139
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max        2395.000000
Name: total_clicks_h1, dtype: float64
zero click rows: 267852
zero click pct: 83.76709886851931

=== Content Type ===
content_type
keyword article       266261
feedly article         50105
comparison article      3392
Name: count, dtype: int64

=== Search Volume ===
null search_volume rows: 58348
zero search_volume rows: 120877
total rows: 319758
count      261410.0
mean     143.186374
std 

## 2. Build the ranked queue (writes the CSV)

Score combines the two validated signals: age bucket (young=3, mid=1, old=0) plus content type (keyword article=2, comparison article=1, feedly article=0), range 0-5. Reason code follows a priority order (young+keyword first, then young alone, then aged+keyword, else low risk). Action from score: 4-5 refresh, 2-3 monitor, 0-1 no_action. Rows with undefined content age are excluded, same as Section 1. Queue is sorted by score and written to work/outputs/baseline_action_score.csv.

In [22]:
# Build the rule on the full frame, excluding rows with undefined age
rule_frame = frame[frame["content_age_days"] >= 0].copy()

rule_frame["age_bucket"] = pd.qcut(rule_frame["content_age_days"], q=3,
                                    labels=["young", "mid", "old"])

age_points = {"young": 3, "mid": 1, "old": 0}
type_points = {"keyword article": 2, "comparison article": 1, "feedly article": 0}

rule_frame["age_points"] = rule_frame["age_bucket"].map(age_points)
rule_frame["type_points"] = rule_frame["content_type"].map(type_points)
rule_frame["score"] = rule_frame["age_points"].astype(float) + rule_frame["type_points"].astype(float)

def assign_reason_code(row):
    if row["age_bucket"] == "young" and row["content_type"] == "keyword article":
        return "YOUNG_KEYWORD_ARTICLE"
    elif row["age_bucket"] == "young":
        return "YOUNG_OTHER_TYPE"
    elif row["content_type"] == "keyword article":
        return "AGED_KEYWORD_ARTICLE"
    else:
        return "LOW_RISK"

rule_frame["reason_code"] = rule_frame.apply(assign_reason_code, axis=1)

rule_frame["action"] = "no_action"
rule_frame.loc[rule_frame["score"] >= 2, "action"] = "monitor"
rule_frame.loc[rule_frame["score"] >= 4, "action"] = "refresh"

ranked_queue = rule_frame.sort_values("score", ascending=False).reset_index(drop=True)

print("=== Ranked Queue ===")
print("rows in ranked queue:", len(ranked_queue))
print(ranked_queue["action"].value_counts())
print(ranked_queue["reason_code"].value_counts())

from pathlib import Path

output_cols = ["content_hash_id", "client_hash_id", "content_age_days", "content_type",
               "score", "reason_code", "action"]
Path("../outputs").mkdir(parents=True, exist_ok=True)
ranked_queue[output_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print("\nwritten to work/outputs/baseline_action_score.csv")

=== Ranked Queue ===
rows in ranked queue: 302692
action
monitor      166124
refresh       95047
no_action     41521
Name: count, dtype: int64
reason_code
AGED_KEYWORD_ARTICLE     158675
YOUNG_KEYWORD_ARTICLE     91660
LOW_RISK                  41524
YOUNG_OTHER_TYPE          10833
Name: count, dtype: int64

written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

All 20 rows tie at the max score (5.0), reason code YOUNG_KEYWORD_ARTICLE, action refresh. 16 belong to one client at content age 4, 3 to a second client at content age 17, and 1 to a third client at content age 69. The rule has no tiebreaker beyond score, so this order is not a real ranking within the group, and there is no per-row information to distinguish them further.

Confidence is high on the category, both signals showed large, validated gaps in the bucket tables, and low on rank within the tie, since score alone cannot separate one young keyword article from another.

What would make the group wrong: any of these pages already showing strong early clicks, ranking position, or search volume. The rule has no visibility into current performance, only age and type, so a page that is young and a keyword article but already succeeding would still be flagged here incorrectly.

In [25]:
top20 = ranked_queue[output_cols].head(20)
top20

,content_hash_id,client_hash_id,content_age_days,content_type,score,reason_code,action
0,content_f3f42ac915539ffd,client_20259bd6705d81d4,69,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
1,content_d0dff76c889de68f,client_62f4a7e64f5e0096,17,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
2,content_ac8663da7484669a,client_62f4a7e64f5e0096,17,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
3,content_39d7361b4945d504,client_62f4a7e64f5e0096,17,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
4,content_75da1537ec9e68c3,client_23a62021009f63c4,4,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
5,content_090b0241e100a47b,client_23a62021009f63c4,4,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
6,content_c8a4730e84f9d4ef,client_23a62021009f63c4,4,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
7,content_210e67c4964e08ed,client_23a62021009f63c4,4,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
8,content_d3a760a8665aaab9,client_23a62021009f63c4,4,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh
9,content_c0d1e41dd5f6016d,client_23a62021009f63c4,4,keyword article,5.0,YOUNG_KEYWORD_ARTICLE,refresh


## 4. Weak picks + leakage check

**Weak picks:** 91,660 of 302,692 rows (30.3%) tie at the max score, all YOUNG_KEYWORD_ARTICLE. A third of the dataset as "top priority" isn't useful triage. Cause: 2 signals, 6 possible score values, too coarse to separate a set this large.

**Leakage check:** only content_age_days and content_type feed the score, both known before H2 exists. is_declining and total_clicks_h2 were never used. No future or label-derived data entered the rule.

**Named limitation:** 5.3% of rows excluded for undefined content age. average_search_position and avg_engagement_rate_h1 unusable, 52.5% and 96.6% missing, which are already established with real numbers in Section 1.

In [24]:
# Weak picks
max_score = ranked_queue["score"].max()
n_at_max = (ranked_queue["score"] == max_score).sum()
print("rows at max score:", n_at_max, "/", len(ranked_queue),
      "=", round(n_at_max / len(ranked_queue) * 100, 1), "%")

# Leakage check
scoring_inputs = ["content_age_days", "content_type"]
print("\nscoring inputs:", scoring_inputs)
print("is_declining in scoring_inputs or output_cols:",
      "is_declining" in scoring_inputs, "/", "is_declining" in output_cols)
print("total_clicks_h2 in scoring_inputs or output_cols:",
      "total_clicks_h2" in scoring_inputs, "/", "total_clicks_h2" in output_cols)

rows at max score: 91660 / 302692 = 30.3 %

scoring inputs: ['content_age_days', 'content_type']
is_declining in scoring_inputs or output_cols: False / False
total_clicks_h2 in scoring_inputs or output_cols: False / False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.